# Análise de Resultados — LLM Performance Lab

Visualização dos CSVs gerados por:
- `results/benchmark/benchmark_resultados.csv` (llama-bench)
- `results/quality/quality_resultados.csv` (avaliação de qualidade)

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

BASE = Path('..').resolve()
BENCHMARK_CSV = BASE / 'results' / 'benchmark' / 'benchmark_resultados.csv'
QUALITY_CSV = BASE / 'results' / 'quality' / 'quality_resultados.csv'
QUALITY_V1 = BASE / 'results' / 'quality' / 'quality_resultados_v1.csv'

print('Benchmark CSV:', BENCHMARK_CSV, '→', BENCHMARK_CSV.exists())
print('Quality CSV:  ', QUALITY_CSV, '→', QUALITY_CSV.exists())
if QUALITY_V1.exists():
    print('(backup v1 com prompts antigos:', QUALITY_V1, ')')

## Benchmark — tokens/s por quantização

In [ ]:
if BENCHMARK_CSV.exists():
    df_bench = pd.read_csv(BENCHMARK_CSV)
    df_bench = df_bench.drop_duplicates(subset=['arquivo'], keep='last')
    display(df_bench)

    df_plot = df_bench.copy()
    df_plot['quant'] = df_plot['arquivo'].str.extract(r'(Q\d+_K[^.]*|F16)', expand=False)
    df_plot['pp512_tps'] = pd.to_numeric(df_plot['pp512_tps'], errors='coerce')
    df_plot['tg128_tps'] = pd.to_numeric(df_plot['tg128_tps'], errors='coerce')
    df_plot = df_plot.dropna(subset=['pp512_tps'])

    if len(df_plot):
        fig, ax = plt.subplots(1, 2, figsize=(12, 4))
        df_plot.plot.bar(x='quant', y='pp512_tps', ax=ax[0], legend=False, color='steelblue')
        ax[0].set_title('Prompt processing (512 tokens)')
        ax[0].set_ylabel('tokens/s')
        df_plot.plot.bar(x='quant', y='tg128_tps', ax=ax[1], legend=False, color='coral')
        ax[1].set_title('Text generation (128 tokens)')
        ax[1].set_ylabel('tokens/s')
        plt.tight_layout()
        plt.show()
    else:
        print('Sem TPS válido. Execute: python scripts/02_benchmark_models.py --timeout 900')
else:
    print('Execute: python scripts/02_benchmark_models.py --timeout 900')

## Qualidade — acurácia por dataset e modelo

In [ ]:
if QUALITY_CSV.exists():
    df_q = pd.read_csv(QUALITY_CSV)
    df_q['acertou'] = df_q['acertou'].astype(str).str.lower().isin(['true', '1', 'yes'])
    df_ok = df_q[df_q['erro'].fillna('') == '']

    if len(df_ok):
        acc = (
            df_ok.groupby(['modelo', 'dataset'])['acertou']
            .mean()
            .reset_index(name='acuracia')
            .sort_values('acuracia', ascending=False)
        )
        display(acc)

        pivot = acc.pivot(index='dataset', columns='modelo', values='acuracia')
        fig, ax = plt.subplots(figsize=(10, max(4, len(pivot) * 0.35)))
        pivot.plot.barh(ax=ax)
        ax.set_xlabel('Acurácia')
        ax.set_title('Acurácia por dataset e modelo')
        plt.tight_layout()
        plt.show()
    else:
        display(df_q.head())
        print('Nenhuma linha sem erro ainda.')
else:
    print('Execute: python scripts/03_quality_eval.py')

## Resumo

In [ ]:
if BENCHMARK_CSV.exists():
    print('Benchmark — modelos:', len(df_bench))
if QUALITY_CSV.exists():
    print('Quality — linhas:', len(df_q), '| acertos:', int(df_q['acertou'].sum()), f"({df_q['acertou'].mean():.1%})")